In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# INGESTION

1. Load documents.
2. Split into chunks.
3. Create a vector store.

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

### Choose LLM you want to work with

In [2]:
# Initialize the local Ollama model
model = ChatOllama(model="llama3", temperature=0)

In [3]:
# Create a simple prompt template
# system_prompt = ("You are a helpful assistant that answers in exactly one sentence.")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers in exactly one sentence."),
    ("human", "{question}")
])

# Chain them together and invoke
chain = prompt | model
response = chain.invoke({"question": "What is the length of asia's longest river?"})

print(response.content)

#system_prompt = (
#    "You are an assistant for question-answering tasks."
#    "Use the following pieces of retrieved context to answer the question."
#    "If you don't know the answer, say that you don't know."
#    "Use three sentences maximum and keep the answer concise."
#    "\n\n"
#    "{context}"
#)


#chat_prompt = ChatPromptTemplate.from_messages(
#    [
#        ("system", system_prompt),
#        ("human", "{input}"),
#    ]
#)

The Yangtze River, also known as the Chang Jiang, is approximately 6,300 kilometers (3,915 miles) long, making it Asia's longest river.


### Step1: Extracting text from PDFs

In [4]:
from langchain_community.document_loaders import PyPDFLoader

/var/folders/ll/_j2y27rj0zz3sty0cy657hsc0000gn/T/ipykernel_1165/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/jyotisingh/RAD_pipeline_2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
pdf_path = "/Users/jyotisingh/Downloads/asia_overview.pdf"

In [6]:
# Load the document
loader = PyPDFLoader(pdf_path)
# Load all pages into Document pages.
docs = loader.load()

In [7]:
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:41:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:41:15+00:00', 'subject': '(unspecified)', 'title': 'Asia Overview', 'trapped': '/False', 'source': '/Users/jyotisingh/Downloads/asia_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Asia: A Continent Profile\nAsia is the Earth's largest and most populous continent, located primarily in the Eastern and Northern\nHemispheres. It shares the continental landmass of Eurasia with Europe and the continental landmass of\nAfro-Eurasia with both Europe and Africa. Covering roughly 30% of Earth's total land area, Asia is home to\nover 60% of the current global human population.\nKey Facts & Demographics\n \x7f Total Area: Approximately 44.6 million square kilometers, making up nearly 9% of Earth's total surface area.\n \x7f Population: Around 4.7 billion 

In [8]:
len(docs) #1
#type(docs) #list
#docs[-1]


1

In [11]:
# docs
# Print the text content of the first page.
print(docs[0].page_content)

Asia: A Continent Profile
Asia is the Earth's largest and most populous continent, located primarily in the Eastern and Northern
Hemispheres. It shares the continental landmass of Eurasia with Europe and the continental landmass of
Afro-Eurasia with both Europe and Africa. Covering roughly 30% of Earth's total land area, Asia is home to
over 60% of the current global human population.
Key Facts & Demographics
  Total Area: Approximately 44.6 million square kilometers, making up nearly 9% of Earth's total surface area.
  Population: Around 4.7 billion people. The two most populous nations on Earth, India and China, drive this
 density.
 Geographical Borders: Bounded by the Arctic Ocean to the north, the Pacific Ocean to the east, and the
 Indian Ocean to the south.
 Continental Boundaries: Separated from Europe traditionally by the Ural Mountains, the Caucasus, and the
 Caspian and Black Seas.
Major Geographical Features
  Highest Point: Mount Everest, standing at 8,848 meters abov

In [9]:
for i,doc in enumerate(docs[:2]): # Show first 2 documents.
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Content length: {len(doc.page_content)} characters")
    print(f"  Content preview: {doc.page_content[:100]}...")
    print(f"  metadata: {doc.metadata}")



Document 1:
  Source: /Users/jyotisingh/Downloads/asia_overview.pdf
  Content length: 1467 characters
  Content preview: Asia: A Continent Profile
Asia is the Earth's largest and most populous continent, located primarily...
  metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:41:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:41:15+00:00', 'subject': '(unspecified)', 'title': 'Asia Overview', 'trapped': '/False', 'source': '/Users/jyotisingh/Downloads/asia_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


## 2. Splitting the documents into Chunks

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
# Split the document into chunks.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:41:15+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T11:41:15+00:00', 'subject': '(unspecified)', 'title': 'Asia Overview', 'trapped': '/False', 'source': '/Users/jyotisingh/Downloads/asia_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Asia: A Continent Profile\nAsia is the Earth's largest and most populous continent, located primarily in the Eastern and Northern\nHemispheres. It shares the continental landmass of Eurasia with Europe and the continental landmass of\nAfro-Eurasia with both Europe and Africa. Covering roughly 30% of Earth's total land area, Asia is home to\nover 60% of the current global human population.\nKey Facts & Demographics"),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-08-31T11:41:15+00:00', '

In [12]:

#Number of chunks created: len(chunks)

print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 4


In [13]:
if chunks:

    for i, chunk in enumerate(chunks[:5]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Source: {chunk.metadata['source']}")
        print(f"Length: {len(chunk.page_content)} characters")
        print(f"Content:")
        print(chunk.page_content)
        print("-" * 50)

if len(chunks) > 5:
    print(f"\n... and {len(chunks) - 5} more chunks")



--- Chunk 1 ---
Source: /Users/jyotisingh/Downloads/asia_overview.pdf
Length: 412 characters
Content:
Asia: A Continent Profile
Asia is the Earth's largest and most populous continent, located primarily in the Eastern and Northern
Hemispheres. It shares the continental landmass of Eurasia with Europe and the continental landmass of
Afro-Eurasia with both Europe and Africa. Covering roughly 30% of Earth's total land area, Asia is home to
over 60% of the current global human population.
Key Facts & Demographics
--------------------------------------------------

--- Chunk 2 ---
Source: /Users/jyotisingh/Downloads/asia_overview.pdf
Length: 438 characters
Content:
over 60% of the current global human population.
Key Facts & Demographics
  Total Area: Approximately 44.6 million square kilometers, making up nearly 9% of Earth's total surface area.
  Population: Around 4.7 billion people. The two most populous nations on Earth, India and China, drive this
 density.
 Geographical Borders: 

In [14]:
chunks[0].metadata # Metadata is created automatically.

{'producer': 'ReportLab PDF Library - www.reportlab.com',
 'creator': '(unspecified)',
 'creationdate': '2026-08-31T11:41:15+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-08-31T11:41:15+00:00',
 'subject': '(unspecified)',
 'title': 'Asia Overview',
 'trapped': '/False',
 'source': '/Users/jyotisingh/Downloads/asia_overview.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}

## 3. Creating Embeddings for the Chunks

Create and persist ChromaDB Vector Store

In [15]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
# Sample LangChain documents
#documents = [
#    Document(page_content="Ollama allows you to run open-source LLMs locally.", metadata={"source": "ollama_guide"}),
#    Document(page_content="Chroma is an open-source embedding database for AI applications.", metadata={"source": "chroma_guide"})
#]

In [17]:
print("Creating embeddings and storing in ChromaDB")

Creating embeddings and storing in ChromaDB


In [18]:
# Embedding model of your choice

embedding_model = OllamaEmbeddings(model = "llama3.2")

## 4. Create and store Embeddings in Local Vector Store

In [19]:
print("Creating Chroma store")

# Create and populate the vector store
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"   
)

Creating Chroma store


In [20]:
retriever = vector_store.as_retriever()

# RETRIEVAL PIPELINE

In [24]:
# Search for relevant documents
query = "what is asia's longest river?"

In [29]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})

In [30]:
relevant_docs = retriever.invoke(query)

In [31]:
relevant_docs

[Document(metadata={'moddate': '2026-08-31T11:41:15+00:00', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'page_label': '1', 'keywords': '', 'page': 0, 'creationdate': '2026-08-31T11:41:15+00:00', 'author': '(anonymous)', 'creator': '(unspecified)', 'title': 'Asia Overview', 'trapped': '/False', 'source': '/Users/jyotisingh/Downloads/asia_overview.pdf', 'total_pages': 1, 'subject': '(unspecified)'}, page_content="over 60% of the current global human population.\nKey Facts & Demographics\n \x7f Total Area: Approximately 44.6 million square kilometers, making up nearly 9% of Earth's total surface area.\n \x7f Population: Around 4.7 billion people. The two most populous nations on Earth, India and China, drive this\n density.\n\x7f Geographical Borders: Bounded by the Arctic Ocean to the north, the Pacific Ocean to the east, and the\n Indian Ocean to the south."),
 Document(metadata={'subject': '(unspecified)', 'moddate': '2026-08-31T11:41:15+00:00', 'title': 'Asia Overview', '

In [32]:
print(f"User Query: {query}")
# Display results
print("--- Context ---")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")


User Query: what is asia's longest river?
--- Context ---
Document 1:
over 60% of the current global human population.
Key Facts & Demographics
  Total Area: Approximately 44.6 million square kilometers, making up nearly 9% of Earth's total surface area.
  Population: Around 4.7 billion people. The two most populous nations on Earth, India and China, drive this
 density.
 Geographical Borders: Bounded by the Arctic Ocean to the north, the Pacific Ocean to the east, and the
 Indian Ocean to the south.

Document 2:
Asia: A Continent Profile
Asia is the Earth's largest and most populous continent, located primarily in the Eastern and Northern
Hemispheres. It shares the continental landmass of Eurasia with Europe and the continental landmass of
Afro-Eurasia with both Europe and Africa. Covering roughly 30% of Earth's total land area, Asia is home to
over 60% of the current global human population.
Key Facts & Demographics



In [ ]:
vector_store.similarity_search("what is asia's population", k=2)

#vector_store.get()

# Perform a basic similarity search returning top k documents.

results = vector_store.similarity_search(query = "what is asia's longest river", k=3)

results


# ANSWER GENERATION

In [22]:
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer the question."
    "If you don't know the answer, say that you don't know."
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

In [23]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [33]:
# Create the document chain (handles formatting the context for the LLM)

question_answering_chain = create_stuff_documents_chain(model, chat_prompt)

In [34]:
# Combine Retriever and Document Chain into the Final Retrieval Chain

rag_chain = create_retrieval_chain(retriever, question_answering_chain)

#retriever = your_vector_store.as_retriever()

In [35]:
# Invoke the chain

response = rag_chain.invoke({"input":"what is asia's longest river?"})

In [36]:
print(response["answer"])

I don't know. The provided context does not mention Asia's longest river.


In [37]:
response = rag_chain.invoke({"input":"What is the length of asia's longest river?"})

In [38]:
print(response["answer"])

I don't know. The provided context does not mention the length of Asia's longest river.


In [39]:
response = rag_chain.invoke({"input":"What is its length?"})

In [40]:
print(response["answer"])

I don't know.


## 5. Semantic Search

In [ ]:
# Chain them together and invoke
chain = prompt | model
response = chain.invoke({"question": "what is asia's longest river?"})

print(response.content)

# prev_ans = response.content

Asia's longest river is the Yangtze River, also known as the Chang Jiang, which stretches approximately 6,300 kilometers (3,915 miles) from its source in Tibet to its mouth in Shanghai, China.


In [34]:
model.invoke("what is asia's longest river?")

AIMessage(content="Asia's longest river is the Yangtze River (also known as the Chang Jiang), which is approximately 6,300 kilometers (3,915 miles) long. It flows through China and empties into the East China Sea.\n\nThe Yangtze River is not only Asia's longest river but also one of the most important rivers in China, supporting a vast array of ecosystems, industries, and human settlements along its length.", additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-09-05T08:01:47.196228Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15713566347, 'load_duration': 27720548, 'prompt_eval_count': 17, 'prompt_eval_duration': 1441000000, 'eval_count': 87, 'eval_duration': 14241000000, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'}, id='lc_run--01a07096-4555-7e53-860e-7cd6ea5332f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 87, 'total_tokens': 104})

In [35]:
context = vector_store.similarity_search(query = "what is asia's longest river", k=3)

In [36]:
model.invoke(f"what is asia's longest river? You can answer using the following context: {context}")

AIMessage(content="Based on the provided context, Asia's longest river is not explicitly mentioned. However, we can infer that the Yangtze River (also known as the Chang Jiang) in China is likely to be one of the longest rivers in Asia, with a length of approximately 6,300 kilometers (3,915 miles).", additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-09-05T08:02:48.822273Z', 'done': True, 'done_reason': 'stop', 'total_duration': 57675527808, 'load_duration': 49457462, 'prompt_eval_count': 762, 'prompt_eval_duration': 45320000000, 'eval_count': 65, 'eval_duration': 12300000000, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'}, id='lc_run--01a07096-9228-7951-ac29-be2bcfa62cfe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 762, 'output_tokens': 65, 'total_tokens': 827})

In [37]:
response = model.invoke(f"what is asia's longest river? You can answer using the following context: {context}")

print(response.content)

Based on the provided context, Asia's longest river is not explicitly mentioned. However, we can infer that the Yangtze River (also known as the Chang Jiang) in China is likely to be one of the longest rivers in Asia, with a length of approximately 6,300 kilometers (3,915 miles).


# HISTORY AWARE GENERATION

## Conversational RAG with Chat History

### History aware retriever

In basic RAG, each query is treated independently. The Retriever takes your exact question and searches for chunks.

In history aware RAG, there's one crucial step: query reformulation.

Before searching, the system looks at the conversation history and rewrites vague or context-dependent questions into clear, standalone questions.

In [43]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


In [44]:
retriever_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

In [45]:
# for the retriever

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", retriever_prompt),
        MessagesPlaceholder(variable_name = "chat_history"),
        ("human", "{input}"),
    ]
)
#chat_prompt = ChatPromptTemplate.from_messages(
#    [
#        ("system", system_prompt),
#        ("human", "{input}"),
#    ]
#)

In [46]:
# Create the history-aware retriever

history_aware_retriever = create_history_aware_retriever(model, retriever, contextualize_q_prompt)

# contextualized prompt | LLM | StrOutputParser | Retriever -> Relevant Document

In [47]:
# for LLM

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [48]:
question_answer_chain = create_stuff_documents_chain(model, qa_prompt)

In [49]:
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [50]:
from langchain_core.messages import HumanMessage,AIMessage 

In [51]:
# Store our conversations as messages
chat_history = []

In [52]:
question1 = "What is Asia's longest river?"

In [53]:
message1 = rag_chain.invoke({"input": question1, "chat_history":chat_history})

In [54]:
message1["answer"]

"I don't know. The provided context does not mention Asia's longest river."

In [55]:
# Remember this conversation

chat_history.extend(
    [
        HumanMessage(content = question1),
        AIMessage(content=message1["answer"]),
    ]
)

In [56]:
chat_history

[HumanMessage(content="What is Asia's longest river?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't know. The provided context does not mention Asia's longest river.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [57]:
chat_history[0]

HumanMessage(content="What is Asia's longest river?", additional_kwargs={}, response_metadata={})

In [58]:
chat_history[1]

AIMessage(content="I don't know. The provided context does not mention Asia's longest river.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])

In [59]:
question2 = "Can you provide its population?"
message2 = rag_chain.invoke({"input":question2, "chat_history":chat_history})

In [60]:
print(message2["answer"])

The provided context does not mention the population of Asia.


In [61]:
# Remember this conversation

chat_history.extend(
    [
        HumanMessage(content = question2),
        AIMessage(content=message2["answer"]),
    ]
)

In [62]:
chat_history

[HumanMessage(content="What is Asia's longest river?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't know. The provided context does not mention Asia's longest river.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you provide its population?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The provided context does not mention the population of Asia.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [63]:
question3 = "Can you tell me more about tourist attractions?"
message3 = rag_chain.invoke({"input":question3, "chat_history":chat_history})

In [64]:
print(message3["answer"])

I don't have information on specific tourist attractions in Asia. The provided context only provides general demographic and geographical information about the continent.


In [65]:
chat_history

[HumanMessage(content="What is Asia's longest river?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't know. The provided context does not mention Asia's longest river.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you provide its population?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The provided context does not mention the population of Asia.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [66]:
# Remember this conversation
chat_history.extend(
    [
        HumanMessage(content = question3 ),
        AIMessage(content=message3["answer"]),
    ]
)

In [67]:
chat_history

[HumanMessage(content="What is Asia's longest river?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't know. The provided context does not mention Asia's longest river.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you provide its population?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The provided context does not mention the population of Asia.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you tell me more about tourist attractions?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't have information on specific tourist attractions in Asia. The provided context only provides general demographic and geographical information about the continent.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [68]:
len(chat_history)

6

In [69]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#### RunnableWithMessageHistory 

-is designed for LCEL (LangChain Expression Language) and modern runnables.

In [70]:
store = {}

**In-Memory Store Implementation**

This is ideal for local development, prototyping, and testing. It stores chat history in a standard Python dictionary.

In [71]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [72]:
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)


/Users/jyotisingh/RAD_pipeline_2/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [73]:
conversational_rag_chain.invoke(
    {"input":"What is Asia's population?"},
    config = {
        "configurable": {"session_id": "abc123"}
    }, # constructs a key "abc123 in 'store"
)['answer']

'Asia is home to over 60% of the current global human population.'

In [74]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content="What is Asia's population?", additional_kwargs={}, response_metadata={}), AIMessage(content='Asia is home to over 60% of the current global human population.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [76]:
for message in store["abc123"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")



User: What is Asia's population?

AI: Asia is home to over 60% of the current global human population.

